In [1]:
from pymarc import MARCReader
import pandas as pd
import requests
from dotenv import load_dotenv
import os
from pinecone import Pinecone
import time
import datetime
import json
import glob

In [2]:
# Load environment variables from .env file
load_dotenv()

# Get ISBN API key from environment variables
ISBN_API_KEY = os.getenv('ISBN_API_KEY')
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

In [3]:
today = datetime.datetime.now()

In [4]:
def extract_book_info(response_json):
    """
    Extract specific fields from ISBN API response and return as dictionary.
    
    Args:
        response_json (dict): The JSON response from the ISBN API
        
    Returns:
        dict: Dictionary containing extracted book information
    """
    extracted_info = {}
    
    # The response might have the book data nested under 'book' key
    book_data = response_json.get('book', response_json)
    
    extracted_info['_id'] = book_data.get('isbn', '')
    
    # Extract synopsis (might be under 'synopsis', 'overview', or 'description')
    extracted_info['synopsis'] = (
        book_data.get('synopsis') or 
        book_data.get('overview') or 
        book_data.get('description') or 
        ''
    )
    
    # Extract title_long
    extracted_info['title'] = book_data.get('title_long', book_data.get('title', ''))
    
    
    # Extract subjects (might be a list or string)
    subjects = book_data.get('subjects')
    if isinstance(subjects, list):
        extracted_info['subjects'] = subjects
    elif isinstance(subjects, str):
        extracted_info['subjects'] = [subjects]
    else:
        extracted_info['subjects'] = ''
    
    # Extract authors (might be a list or string)
    authors = book_data.get('authors')
    if isinstance(authors, list):
        extracted_info['authors'] = authors
    elif isinstance(authors, str):
        extracted_info['authors'] = [authors]
    else:
        extracted_info['authors'] = ''
    
    return extracted_info


In [5]:
def get_book_info(isbn):
    # https://isbndb.com/user/62794
    if isbn is None:
        return None

    # Clean the ISBN (remove any extra characters, spaces, etc.)
    clean_isbn = isbn.replace('-', '').replace(' ', '').strip()
    
    # API request to ISBN database
    url = f"https://api2.isbndb.com/book/{clean_isbn}"
    headers = {
        'User-Agent': 'python-requests/2.28.1',
        'Authorization': ISBN_API_KEY,  # Replace with your actual API key
        'Accept': '*/*'
    }
    
    
    try:
        start_time = time.time()
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            book_info = extract_book_info(response.json())
            end_time = time.time()
            if end_time - start_time > 1:
                return book_info
            else:
                time.sleep(1)
                return book_info
        else:
            print(f"Error Response: {response.text}")
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")


In [26]:
def check_none_values(record_dict):
    """
    Check each item in a single record dictionary for None values.
    Replaces None values with empty strings and prints what was changed.
    
    Args:
        record_dict (dict): A single record dictionary to check and modify
        
    Returns:
        dict: The modified dictionary with None values replaced by empty strings
    """
    none_count = 0
    
    try:
        for key, value in record_dict.items():
            if value is None or value == '' or value == 'null':
                record_dict[key] = '-999'  # Replace None with empty string
            none_count += 1
    except Exception as e:
        print(f"Error checking record: {e}")
        return None

    return record_dict

In [7]:
# Path to your MARC file
marc_file = 'ExportBibJob605495USMarc.001'

In [8]:
existing_records = {}

# Get all JSON files in the data folder
json_files = glob.glob('data/book_records_*.json')

# Load and combine all records, using dict to automatically handle duplicates
for file in json_files:
    try:
        file_records = json.load(open(file))
        # Convert list of records to dict keyed by _id
        for r in file_records:
            if r is None:
                continue
            else:
                existing_records[r['_id']] = r
    except Exception as e:
        print(f"Error loading {file}: {e}")

# Convert back to list for consistency with rest of code
existing_records = list(existing_records.values())
print(f"Loaded {len(existing_records)} unique records")

Loaded 6373 unique records


In [44]:
def clean_text(text):
    text = text.replace(' /', '')
    text = text.replace('.', '')
    return text

In [45]:
books = []
isbn_errors = []
true_errors = 0
with open(marc_file, 'rb') as file:
    reader = MARCReader(file, to_unicode=True, force_utf8=True)
    for record in reader:
        try:
            isbn = record['020']['a'] if record['020'] else ''
            authors = record['100']['a'] if record['100'] else ''
            title = record['245']['a'] if record['245'] else ''
            summary = record['520']['a'] if record['520'] else '' 

            subjects = []
            for subject_field in record.get_fields('650'):
                if subject_field['a']:
                    subjects.append(clean_text(subject_field['a'] if subject_field['a'] else ''))

            books.append({
                '_id': isbn,
                'authors': clean_text(authors),
                'title': clean_text(title),
                'synopsis': summary,
                'subjects': subjects
            })
                
        except:
            try:
                isbn_errors.append(record['020']['a'] if record['020'] else '')
            except:
                true_errors += 1

print("Number of books: ", len(books))
print("Number of extracted ISBNs: ", len(isbn_errors))
print("Number of true errors: ", true_errors)

Number of books:  10642
Number of extracted ISBNs:  631
Number of true errors:  8


In [11]:
# List to store records
records = []
num_errors = 0

for isbn in isbn_errors:
    try:
        book_info = get_book_info(isbn)
        if book_info is None:
            continue
        else:
            book_info_cleaned = check_none_values(book_info)
            # Add the record to the list
            records.append(book_info_cleaned)
        
    except Exception as e:
        print(f"Error processing record {book_info.get('isbn', 'unknown')}: {e}")
        num_errors += 1

    if len(records) >= 4000:
        break

print(f"Number of records processed: {len(records)}")
print(f"Number of errors: {num_errors}")

Error Response: {"errorType":"string","errorMessage":"Not Found","trace":[]}
Error Response: {"errorType":"string","errorMessage":"Not Found","trace":[]}
Error Response: {"errorType":"string","errorMessage":"Not Found","trace":[]}
Error Response: {"errorType":"string","errorMessage":"Not Found","trace":[]}
Number of records processed: 627
Number of errors: 0


In [12]:
# Create filename with date
filename = 'data/error_records_{}.json'.format(today.strftime('%Y-%m-%d'))

# Save records to JSON file
with open(filename, 'w') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} records to {filename}")

Saved 627 records to data/error_records_2025-08-13.json


In [47]:
# Combine clean_records and books into one full list
all_records = records + books

print(f"Total number of records: {len(all_records)}")

Total number of records: 11269


In [48]:
clean_records = []
for r in all_records:
    new_r = check_none_values(r)
    if new_r is not None:
        clean_records.append(new_r)

print(len(clean_records))

11269


In [15]:
index_name = "whitman"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map": {
                "text": "title",
                "text": "authors",
                "text": "subjects",
                "text": "synopsis",
            }
        }
    )

In [16]:
# Target the index
dense_index = pc.Index(index_name)

/Users/emmettstorts/.local/share/virtualenvs/destiny-rag-yAKJ2ugu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [52]:
problem_records = []
count = 0
limit = 90
for i in range(0, len(all_records), limit):
    try:
        if count < limit:
            dense_index.upsert_records("whitman", all_records[i:i+limit])
            count += 1
        else:
            count = 0
            print(f"Sleeping for 60 seconds")
            time.sleep(60)
    except Exception as e:
        print(f"Error upserting records: {e}")
        print(f"Count: {count}")
        problem_records.append(all_records[i:i+limit])

Sleeping for 60 seconds


In [53]:
problem_records_again = []
count = 0
for i in problem_records:
    try:
        if count < 5:
            dense_index.upsert_records("whitman", i)
            count += 1
        else:
            count = 0
            print(f"Sleeping for 60 seconds")
            time.sleep(60)
    except Exception as e:
        print(f"Error upserting records: {e}")
        print(f"Count: {count}")
        problem_records_again.append(i)

if len(problem_records_again) > 0:
    print(f"There are {len(problem_records_again)} records that need to be upserted again")
else:
    print("All records have been upserted successfully")

All records have been upserted successfully


In [ ]:
time.sleep(10)

# View stats for the index
stats = dense_index.describe_index_stats()
print(stats) 

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'whitman': {'vector_count': 11252}},
 'total_vector_count': 11252,
 'vector_type': 'dense'}


In [18]:
query = "what is a book that has to do with the loch ness monster?"

In [22]:
# Search the dense index and rerank results
reranked_results = dense_index.search(
    namespace="example-namespace",
    query={
        "top_k": 3,
        "inputs": {
            'text': query
        }
    },
    rerank={
        "model": "bge-reranker-v2-m3",
        "top_n": 3,
        "rank_fields": ["synopsis"]
    }   
)

# Print the reranked results
for hit in reranked_results['result']['hits']:
    hit_dict = hit.to_dict()
    print(hit_dict)

{'_id': '1669072150', '_score': 0.4333818256855011, 'fields': {'authors': ['Benjamin Harper', 'Laurie S. Sutton', 'Michael Dahl', 'Megan Atwood'], 'subjects': ["Children's Books", 'Literature & Fiction', 'Short Story Collections', 'Science Fiction & Fantasy', 'Spine-Chilling Horror', 'Mysteries & Detectives', 'Short Stories & Anthologies', 'Anthologies'], 'synopsis': "A legendary collection of cryptid stories for young horror fans! Believe it or not . . . the stories in this book will SCARE YOUR SOCKS OFF! Will the Loch Ness monster upend a young boy's Scottish vacation? What happens when the Jersey Devil terrorizes a summer camp? Is Bigfoot lurking outside a remote cabin in the woods? Discover the answers to these creepy questions and more in this spine-chilling collection of eight cryptid stories . . . if you dare! Featuring four scream-worthy horror writers--including bestselling author Michael Dahl--this anthology is an absolute must-have for your next campout, sleepover, or Hallow